# BEB-La-DII: Phase 1 Training (Awakening)
Этот блокнот предназначен для запуска первой фазы дистилляции в среде Kaggle. Он включает автоматическое клонирование репозитория, настройку путей, безопасную установку зависимостей и создание связей с датасетами ресурсов.

In [ ]:
# 0. ПОЛУЧЕНИЕ ИСХОДНОГО КОДА
import os
import sys

REPO_URL = "https://github.com/Laeryid/BEBLaDII"
REPO_NAME = "BEBLaDII"

if not os.path.exists(REPO_NAME):
    print(f"Клонирование репозитория {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Репозиторий {REPO_NAME} уже присутствует.")

# Переход в директорию проекта
if os.path.exists(REPO_NAME) and REPO_NAME not in os.getcwd():
    os.chdir(REPO_NAME)
    print(f"Рабочая директория изменена на: {os.getcwd()}")
else:
    print(f"Текущая рабочая директория: {os.getcwd()}")

In [ ]:
# 1. БЕЗОПАСНАЯ УСТАНОВКА ЗАВИСИМОСТЕЙ
import subprocess
import sys

def install_packages():
    # Фиксируем версии из pyproject.toml для стабильности
    packages = [
        "transformers==4.57.2",
        "indexed-parquet-dataset",
        "optimum-intel[openvino]",
        "wandb",
        "accelerate"
    ]
    print("--- Проверка и установка пакетов ---")
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", pkg])
        print(f"Установлено/Проверено: {pkg}")

install_packages()

# 2. НАСТРОЙКА ПУТЕЙ ИМПОРТА
from pathlib import Path

current_path = Path.cwd()
project_root = None

for path in [current_path] + list(current_path.parents):
    if (path / "src").exists():
        project_root = path
        break

if project_root:
    root_str = str(project_root.absolute())
    print(f"Корень проекта найден: {root_str}")
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
else:
    print("WARNING: Директория 'src' не найдена.")

In [ ]:
import os
import torch
import json
from torch.optim import AdamW
from tqdm.auto import tqdm
import wandb
import shutil

# Импорты проекта
try:
    from src.beb_la_dii.model.distiller import ReasoningDistiller
    from src.beb_la_dii.model.assembler import ModelAssembler
    from src.beb_la_dii.utils.tokenizer import get_tokenizer
    from src.beb_la_dii.utils.loss import DistillationLoss
    from src.beb_la_dii.utils.data import get_dataloader
    from src.beb_la_dii.utils.experiment_tracker import ExperimentTracker
    print("Модули проекта успешно импортированы.")
except ImportError as e:
    print(f"Ошибка импорта: {e}")

In [ ]:
# КОНСТАНТЫ И ГИПЕРПАРАМЕТРЫ
BASE_MODEL_NAME = "answerdotai/ModernBERT-large"
TEACHER_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
EPOCHS = 1
LEARNING_RATE = 5e-5
STAGE = 'awakening' 
VERSION = "v1.0"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
def setup_kaggle():
    """Настройка инфраструктуры Kaggle: симлинки данных, весов и создание конфигов"""
    if not os.path.exists("/kaggle/input"):
        print("Вне среды Kaggle. Пропуск настройки.")
        return None
        
    print("--- Настройка ресурсов Kaggle ---")
    input_base = "/kaggle/input"
    available_inputs = os.listdir(input_base)
    resource_ds = None
    
    # Поиск датасета ресурсов
    for ds_name in available_inputs:
        search_terms = ["bebladii", "resources", "phase1-resources"]
        if any(term in ds_name.lower() for term in search_terms):
            resource_ds = os.path.join(input_base, ds_name)
            print(f"Найден датасет ресурсов: {ds_name}")
            break
            
    if not resource_ds:
        print("WARN: Датасет ресурсов не найден.")
        return None

    # 1. СТРУКТУРА ДЛЯ ДЛЯ ВЕСОВ И КОНФИГА (latentBERT)
    weights_target_dir = "storage/components/model/latentBERT/v1.0"
    os.makedirs(weights_target_dir, exist_ok=True)
    
    # Создание config.json (КРИТИЧЕСКИ ВАЖНО)
    config_path = os.path.join(weights_target_dir, "config.json")
    if not os.path.exists(config_path):
        print(f"Генерация метаданных: {config_path}")
        config_data = {
            "component_id": "latentBERT",
            "version": "v1.0",
            "config": {
                "base_model_id": "answerdotai/ModernBERT-large",
                "target_layers": 40
            }
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config_data, f, indent=4)
            
    # Симлинк для весов
    src_weights = os.path.join(resource_ds, "weights.pt")
    dst_weights = os.path.join(weights_target_dir, "weights.pt")
    
    if os.path.exists(src_weights) and not os.path.exists(dst_weights):
        print(f"Linking weights: {dst_weights} -> {src_weights}")
        os.symlink(src_weights, dst_weights)
    elif os.path.exists(dst_weights):
        print(f"Файл весов уже присутствует в {dst_weights}")

    # 2. СИМЛИНКИ ДЛЯ ДАННЫХ (Awakening, Reasoning)
    os.makedirs("data", exist_ok=True)
    ds_data_path = os.path.join(resource_ds, "data")
    if os.path.exists(ds_data_path):
        for folder in ["Awakening", "Reasoning"]:
            src = os.path.join(ds_data_path, folder)
            dst = os.path.join("data", folder)
            if os.path.exists(src) and not os.path.exists(dst):
                print(f"Linking data directory: {dst}")
                os.symlink(src, dst)
        
    return resource_ds

resource_ds = setup_kaggle()

In [ ]:
# ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Инициализация на {device}...")

alt_roots = [resource_ds] if resource_ds else None
assembler = ModelAssembler(alt_roots=alt_roots)

distiller = assembler.assemble_phase1_distiller(
    teacher_id=TEACHER_NAME,
    device_map="auto"
)

tracker = ExperimentTracker(project_root=".", stage=STAGE)

In [ ]:
# НАСТРОЙКА ГРАДИЕНТОВ
for param in distiller.parameters():
    param.requires_grad = False
for param in distiller.student.parameters():
    param.requires_grad = True
for param in distiller.input_projector.parameters():
    param.requires_grad = True
for proj in distiller.feature_projectors.values():
    for param in proj.parameters():
        param.requires_grad = True
        
print(f"Обучаемых параметров: {sum(p.numel() for p in distiller.parameters() if p.requires_grad):,}")

In [ ]:
# ПОДГОТОВКА ДАННЫХ
dataloader = get_dataloader(stage=STAGE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
print(f"Данные загружены. Батчей: {len(dataloader)}")

In [ ]:
# ТРЕНИРОВКА
optimizer = AdamW(filter(lambda p: p.requires_grad, distiller.parameters()), lr=LEARNING_RATE)
criterion = DistillationLoss()

if os.environ.get("WANDB_API_KEY"):
    wandb.init(project="beb-la-dii-phase1", name=f"latentbert-{STAGE}-{VERSION}")

distiller.train()
progress_bar = tqdm(dataloader, desc=f"Training Phase 1 ({STAGE})")

accum_loss = 0.0
optimizer.zero_grad()

for step, batch in enumerate(progress_bar):
    input_ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
    
    student_states, teacher_targets = distiller(input_ids, mask)
    loss = criterion(student_states, teacher_targets) / GRAD_ACCUM_STEPS
    
    loss.backward()
    accum_loss += loss.item()
    
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(distiller.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        avg_loss = accum_loss * GRAD_ACCUM_STEPS
        progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
        if wandb.run: wandb.log({"loss": avg_loss, "step": step})
        tracker.log_step(step, {"loss": avg_loss})
        accum_loss = 0.0

In [ ]:
# ЗАВЕРШЕНИЕ
state_to_save = {
    "latentBERT_state_dict": distiller.student.state_dict(),
    "input_projector": distiller.input_projector.state_dict(),
    "feature_projectors": distiller.feature_projectors.state_dict()
}
tracker.save_checkpoint(state_to_save, name=f"latentbert_{STAGE}_final")
tracker.finish()
if wandb.run: wandb.finish()
print("Готово!")